In [173]:
from pathlib import Path

# ========== 根路径与任务设置 ==========
ROOT = Path("./results")          # ./results/gcd/*/results.csv, ./results/openset/*/results.csv
SETTING = "gcd"               # "gcd" 或 "openset"

OUT_DIR = Path(f"./latex_or_csv/{SETTING}")   # 改成你想要的路径
OUT_DIR.mkdir(parents=True, exist_ok=True)


# 每个 setting 对应要读取的任务目录；留空 [] 或 ["*"] 表示扫描该 setting 下所有含 results.csv 的目录
TASKS_MAP = {
    "gcd":     ["alup", "deepaligned", "dpn", "geoid", "loop", "sdc", "tan"],
    "openset": ["ab", "adb", "deepunk", "doc", "dyen", "knncon", "plm_ood", "scl"],
}

# method 字段/目录名的重命名（归一化后匹配）
METHOD_RENAME = {
    # gcd
    # "deepaligned": "DeepAligned",
    "deepaligned": "DAL",
    "geoid": "GeoID",
    "sdc": "SDC",
    "dpn": "DPN",
    "tan": "TAN",
    "loop": "LOOP",
    "alup": "ALUP",

    # openset
    "doc": "DOC",
    "deepunk": "DeepUNK",
    "adb": "ADB",
    "scl": "SCL",
    "ab": "AB",
    "knncon": "KNNCon",
    "knn_con": "KNNCon",
    "dyen": "DyEn",
    "plm_ood": "LLM_OOD",   # 只是兜底名；真正会拆 Detector
}

# Method 显示名（可选）
METHOD_DISPLAY = {
    "DeepAligned": "DAL",
}

# 指标
METRICS_MAP = {
    "gcd":     ["K-ACC", "N-ACC", "H-Score", "ARI", "NMI", "ACC"],
    "openset": ["K-F1", "N-F1", "F1", "ACC"],
}

# 固定 LCR（主实验固定这个）
LCR_KEEP = {"gcd": 1.0, "openset": 0.1}

# KCR 顺序
KCR_ORDER = [0.25, 0.50, 0.75]

# 四舍五入
ROUND_DECIMALS = 2

# 是否高亮 top1/top2
HIGHLIGHT_TOP2 = True

# 只显示 METHOD_ORDER_MAP 里列出的（白名单 + 顺序）
METHOD_ORDER_MAP = {
    # "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
    "gcd": ["DAL", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
    "openset": [
        "DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn",
        "LLM-MaxSoftmax", "LLM-OpenMax", "LLM-TempScale", "LLM-Energy",
        "LLM-LogitNorm", "LLM-Entropy", "LLM-KLMatching", "LLM-MaxLogit",
    ],
}

# ========== 基础数据集列顺序（11个）与显示名（表头）==========
# 注意：这里的 key 用于“匹配/排序”，显示名用于 LaTeX 表头
DATASETS_ORDER = [
    "20NG",
    "thucnews",
    "Yahoo",
    "TREC",
    "banking",
    "stackoverflow",
    "clinc",
    "hwu",
    "ecdt",
    "mcid",
    "DBPedia",
    # "X-Topic"
]

DATASET_DISPLAY = {
    "20NG": "20NG",
    "thucnews": "THUCNews",
    "Yahoo": "Yahoo",
    "TREC": "TREC",
    "banking": "BANK",
    "stackoverflow": "S.O.",
    "clinc": "CLINC",
    "hwu": "HWU",
    "ecdt": "ECDT",
    "mcid": "M-CID",
    "DBPedia": "DBPedia",
    # "X-Topic": "X-Topic"
}

# ========== 你要“额外追加到最后”的数据集（可多个，按顺序追加） ==========
# 例：["X-Topics"]；以后要加别的，就改这里即可
APPEND_DATASETS = ["X-Topic"]

# ========== 其它过滤 ==========
ALIASES_PLM = {"plmood", "llmood"}   # plm_ood 目录别名
FOLD_TYPE_KEEP = "fold"             # 只保留 args.fold_type == "fold"

# 是否复用已有“宽表 pivot CSV”
USE_EXISTING_PIVOT_CSV = True
PIVOT_CSV_PATH = "./exp_results/table/gcd/table_s31.csv" # 现有宽表（列：Metric,KCR,Method,11 datasets, Avg.）

# ========== 输出 ==========
KEEP_LCR_VALUE = LCR_KEEP[SETTING]
OUTPUT_TEX = str(OUT_DIR / f"{SETTING}_table_s31.tex")

# 是否导出“更新后的宽表 CSV”（把追加列 + 新 Avg 写出来）
WRITE_UPDATED_WIDE_CSV = True
UPDATED_WIDE_CSV = str(OUT_DIR / f"{SETTING}_table_s31_plus.csv")


In [174]:
import pandas as pd
import numpy as np
import json
import ast


# ------------------ 工具函数 ------------------
def parse_args_field(s: str) -> dict:
    """尽量鲁棒地把CSV里的args字符串解析成dict。"""
    if pd.isna(s):
        return {}
    txt = str(s)

    try:
        return json.loads(txt)
    except Exception:
        pass

    # 常见转义：{""k"":""v""}
    try:
        fixed = txt.replace('""', '"')
        if len(fixed) >= 2 and fixed[0] == '"' and fixed[-1] == '"':
            fixed = fixed[1:-1]
        return json.loads(fixed)
    except Exception:
        pass

    try:
        return ast.literal_eval(txt)
    except Exception:
        return {}

def normalize_detector_name(det: str) -> str:
    if det is None:
        return "LLM-OOD"
    det = str(det).strip()

    # 已经带 LLM- 前缀就不动
    if det.lower().startswith("llm-"):
        return det

    # 常见别名/统一命名（按你实际情况增补）
    alias = {
        "EnergyBased": "Energy",
        "TemperatureScaling": "TempScale",
    }
    det2 = alias.get(det, det)

    # 统一加前缀，和你的表里 LLM-xxx 对齐
    return f"LLM-{det2}"


def _normalize_key(s: str) -> str:
    return str(s).strip().lower().replace("_", "").replace("-", "")


def _norm_ds(x: str) -> str:
    """dataset 名归一化：忽略大小写、下划线、连字符。"""
    return str(x).strip().lower().replace("_", "").replace("-", "")


def rename_method(raw_name: str, fallback_task: str) -> str:
    key = _normalize_key(raw_name if pd.notnull(raw_name) else fallback_task)
    return METHOD_RENAME.get(key, str(raw_name if pd.notnull(raw_name) else fallback_task))


def method_display_name(m: str) -> str:
    return METHOD_DISPLAY.get(m, m)


def latex_escape_text(x) -> str:
    s = str(x)
    rep = {
        "\\": r"\textbackslash{}",
        "&":  r"\&",
        "%":  r"\%",
        "$":  r"\$",
        "#":  r"\#",
        "_":  r"\_",
        "{":  r"\{",
        "}":  r"\}",
        "~":  r"\textasciitilde{}",
        "^":  r"\textasciicircum{}",
    }
    return "".join(rep.get(ch, ch) for ch in s)


def fmt_kcr(x: float) -> str:
    try:
        return f"{float(x):g}"
    except Exception:
        return str(x)


def ensure_method_whitelist(df: pd.DataFrame) -> pd.DataFrame:
    allowed = METHOD_ORDER_MAP.get(SETTING, []) or []
    if not allowed:
        return df
    df = df[df["Method"].astype(str).isin(set(allowed))].copy()
    if df.empty:
        raise ValueError(f"{SETTING}: 方法白名单过滤后无数据。请检查 METHOD_ORDER_MAP['{SETTING}']")
    return df


def read_tasks_results() -> pd.DataFrame:
    """读取 ./results/{SETTING}/{task}/results.csv，返回合并后的 df（未做 dataset 白名单过滤）。"""
    base = ROOT / SETTING
    task_list = TASKS_MAP.get(SETTING, [])
    if (not task_list) or (task_list == ["*"]):
        task_list = sorted(p.name for p in base.iterdir() if p.is_dir() and (p / "results.csv").exists())

    frames = []
    for task in task_list:
        csv_path = base / task / "results.csv"
        try:
            tdf = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")
        except Exception:
            continue
        if "args" not in tdf.columns:
            continue

        tdf["__args_obj__"] = tdf["args"].apply(parse_args_field)
        tdf["__fold_type__"] = tdf["__args_obj__"].apply(
            lambda d: d.get("fold_type") or d.get("foldType") or d.get("fold-type")
        )
        tdf = tdf[tdf["__fold_type__"].astype(str).str.lower() == str(FOLD_TYPE_KEEP).lower()].copy()
        if tdf.empty:
            continue

        # Method 统一（含 plm_ood -> Detector）
        if "method" in tdf.columns:
            def _method_from_row(row):
                raw_m = row.get("method", None)
                renamed = rename_method(raw_m, task)
                if any(_normalize_key(x) in ALIASES_PLM for x in (task, raw_m, renamed)):
                    det = row["__args_obj__"].get("Detector") or row["__args_obj__"].get("Detecor")
                    # return det if det else "LLM-OOD"
                    return normalize_detector_name(det)
                return renamed
            tdf["Method"] = tdf.apply(_method_from_row, axis=1)
        else:
            if _normalize_key(task) in ALIASES_PLM:
                tdf["Method"] = tdf["__args_obj__"].apply(lambda d: normalize_detector_name(d.get("Detector") or d.get("Detecor")))
            else:
                tdf["Method"] = rename_method(task, task)

        tdf["dataset"] = tdf["dataset"].astype(str)
        tdf["KCR"] = pd.to_numeric(tdf.get("known_cls_ratio", np.nan), errors="coerce")
        tdf["LCR"] = pd.to_numeric(tdf.get("labeled_ratio", np.nan), errors="coerce")

        tdf.drop(columns=["__args_obj__", "__fold_type__"], inplace=True, errors="ignore")
        frames.append(tdf)

    if not frames:
        raise FileNotFoundError("未找到任何可用 results.csv（或全被跳过）。")

    df = pd.concat(frames, ignore_index=True)
    return df


def pivot_from_results(df: pd.DataFrame, datasets_keep: list[str]) -> pd.DataFrame:
    """从 results 合并后的 df 生成 pivot（仅保留 datasets_keep 这些数据集）。"""
    metrics = METRICS_MAP[SETTING]

    df = ensure_method_whitelist(df)

    # 过滤 LCR + KCR
    df = df[df["LCR"] == KEEP_LCR_VALUE].copy()
    df = df[df["KCR"].isin(KCR_ORDER)].copy()
    if df.empty:
        raise ValueError(f"{SETTING}: 过滤 LCR={KEEP_LCR_VALUE}, KCR in {KCR_ORDER} 后无数据。")

    # openset: <1 -> *100（只对 metrics）
    if SETTING == "openset":
        for m in metrics:
            if m in df.columns:
                df[m] = df[m].apply(
                    lambda v: (v * 100.0)
                    if (pd.notnull(v) and isinstance(v, (int, float)) and v < 1.0)
                    else v
                )

    # dataset 白名单（大小写/下划线/连字符宽松匹配）
    keep_keys = set(_norm_ds(x) for x in datasets_keep)
    df = df[df["dataset"].apply(_norm_ds).isin(keep_keys)].copy()
    if df.empty:
        raise ValueError("数据集筛选后无数据。请检查 datasets_keep 与 results.csv 的 dataset 命名。")

    required_cols = ["dataset", "Method", "KCR", "LCR"] + metrics
    miss = [c for c in required_cols if c not in df.columns]
    if miss:
        raise KeyError(f"缺失列：{miss}")

    grouped = (
        df[required_cols]
        .groupby(["dataset", "Method", "KCR", "LCR"], as_index=False)
        .mean(numeric_only=True)
    )

    long = grouped.melt(
        id_vars=["dataset", "Method", "KCR", "LCR"],
        value_vars=metrics,
        var_name="Metric",
        value_name="value"
    )

    pivot = long.pivot_table(
        index=["Metric", "KCR", "Method"],
        columns="dataset",
        values="value",
        aggfunc="mean"
    )
    return pivot


def pivot_from_existing_wide_csv(path: str):
    wide = pd.read_csv(path)
    wide.columns = [c.strip() for c in wide.columns]

    # 统一 Avg 列名
    if "Avg." in wide.columns and "__AVG__" not in wide.columns:
        wide = wide.rename(columns={"Avg.": "__AVG__"})
    if "Avg" in wide.columns and "__AVG__" not in wide.columns:
        wide = wide.rename(columns={"Avg": "__AVG__"})

    for k in ["Metric", "KCR", "Method"]:
        if k not in wide.columns:
            raise ValueError(f"{path} 缺少列: {k}")

    # ✅ 关键：保留“宽表里原本的列顺序/列名”（不做任何改名）
    base_cols = [c for c in wide.columns if c not in ["Metric", "KCR", "Method", "__AVG__"]]

    wide["KCR"] = pd.to_numeric(wide["KCR"], errors="coerce")
    for c in base_cols + (["__AVG__"] if "__AVG__" in wide.columns else []):
        wide[c] = pd.to_numeric(wide[c], errors="coerce")

    pivot = wide.set_index(["Metric", "KCR", "Method"]).sort_index()
    return pivot, base_cols



def extract_one_dataset_column(dataset_name: str) -> pd.DataFrame:
    """
    从 results.csv 里只抽取一个 dataset 的结果，返回：
      index=(Metric,KCR,Method), columns=[dataset_name]
    """
    df_all = read_tasks_results()
    p = pivot_from_results(df_all, datasets_keep=[dataset_name])

    # results 里该 dataset 列名可能大小写不同，这里统一重命名
    real_col = p.columns[0]
    out = p[[real_col]].rename(columns={real_col: dataset_name})
    return out


def align_columns_case_insensitive(pivot: pd.DataFrame, desired_keys: list[str]) -> list[str]:
    """
    根据 desired_keys（例如 DATASETS_ORDER 里的 key）从 pivot.columns 中找匹配列，
    匹配规则：对 pivot 里的列名做 _norm_ds 后与 desired_keys 的 _norm_ds 比较。
    返回 pivot 的“真实列名”列表，按 desired_keys 的顺序排列（只返回存在的）。
    """
    col_map = {}
    for c in pivot.columns:
        if str(c) == "__AVG__":
            continue
        col_map[_norm_ds(c)] = c

    out = []
    for k in desired_keys:
        kk = _norm_ds(k)
        if kk in col_map:
            out.append(col_map[kk])
    return out


def sort_pivot_index(pivot: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """按 Metric/KCR/Method 白名单顺序重排 index，并返回 method_order。"""
    idx = pivot.index
    metrics = METRICS_MAP[SETTING]

    metric_cat = pd.Categorical(idx.get_level_values(0), categories=metrics, ordered=True)
    kcr_cat = pd.Categorical(idx.get_level_values(1), categories=KCR_ORDER, ordered=True)

    method_whitelist_order = METHOD_ORDER_MAP.get(SETTING, []) or []
    methods_present = list(idx.get_level_values(2).unique())
    method_order = [m for m in method_whitelist_order if m in methods_present]
    method_cat = pd.Categorical(idx.get_level_values(2), categories=method_order, ordered=True)

    new_index = pd.MultiIndex.from_arrays([metric_cat, kcr_cat, method_cat], names=idx.names)
    pivot = pivot.reindex(new_index).sort_index(level=[0, 1, 2])
    return pivot, method_order


# ------------------ 构造基表 pivot ------------------
if USE_EXISTING_PIVOT_CSV:
    pivot, base_real_cols = pivot_from_existing_wide_csv(PIVOT_CSV_PATH)
else:
    df_all = read_tasks_results()
    pivot = pivot_from_results(df_all, datasets_keep=DATASETS_ORDER)
    base_real_cols = align_columns_case_insensitive(pivot, DATASETS_ORDER)  # 只有这条链路才需要匹配

# ------------------ 追加数据集列：按 APPEND_DATASETS 顺序追加到最后 ------------------
append_cols = []
for ds_name in APPEND_DATASETS:
    col_df = extract_one_dataset_column(ds_name)
    pivot = pivot.join(col_df, how="left")
    append_cols.append(ds_name)

# ✅ 最终列顺序：先原宽表列（BANK/S.O. 等不会丢），再追加列
final_ds_cols = base_real_cols + append_cols

# 重排 + 重算 Avg
pivot = pivot.reindex(columns=final_ds_cols + (["__AVG__"] if "__AVG__" in pivot.columns else []))
pivot["__AVG__"] = pivot[final_ds_cols].mean(axis=1, numeric_only=True)
pivot = pivot.round(ROUND_DECIMALS)

# 行排序保持（不变）
pivot, method_order = sort_pivot_index(pivot)

# 可选：导出更新后的宽表 CSV
if WRITE_UPDATED_WIDE_CSV:
    wide_out = pivot.reset_index().rename(columns={"__AVG__": "Avg."})
    wide_out.to_csv(UPDATED_WIDE_CSV, index=False, encoding="utf-8")
    print(f"[OK] updated wide csv -> {UPDATED_WIDE_CSV}")


# ------------------ 高亮：每个 dataset/Avg 列内，按 (Metric,KCR) 对 Method 排名 ------------------
def highlight_one_col(col_ser: pd.Series) -> pd.Series:
    out = col_ser.astype(object)
    num = pd.to_numeric(col_ser, errors="coerce")

    for (metric_val, kcr_val), sub in num.groupby(level=[0, 1]):
        order = sub.sort_values(ascending=False, kind="mergesort")
        if len(order) >= 1 and pd.notnull(order.iloc[0]):
            out.loc[(metric_val, kcr_val, order.index.get_level_values(2)[0])] = (
                "\\textbf{" + f"{order.iloc[0]:.{ROUND_DECIMALS}f}" + "}"
            )
        if HIGHLIGHT_TOP2 and len(order) >= 2 and pd.notnull(order.iloc[1]):
            out.loc[(metric_val, kcr_val, order.index.get_level_values(2)[1])] = (
                "\\underline{" + f"{order.iloc[1]:.{ROUND_DECIMALS}f}" + "}"
            )

        for idxi, v in sub.items():
            if pd.isnull(v):
                out.loc[idxi] = ""
            elif out.loc[idxi] == col_ser.loc[idxi]:
                out.loc[idxi] = f"{v:.{ROUND_DECIMALS}f}"
    return out


formatted = pd.DataFrame(index=pivot.index)
for c in list(pivot.columns):
    formatted[c] = highlight_one_col(pivot[c])


# ------------------ 生成 LaTeX（tabularx + multirow + cmidrule + Avg） ------------------
# 表头显示名：基础 key 用 DATASET_DISPLAY；追加列默认显示为其本名（也可自行在 DATASET_DISPLAY 里补映射）
def dataset_header_name(col_real_name: str) -> str:
    # 尝试用 key 映射：把真实列名归一化后反查 DATASETS_ORDER 的 key
    norm = _norm_ds(col_real_name)
    for k in DATASETS_ORDER:
        if _norm_ds(k) == norm:
            return DATASET_DISPLAY.get(k, str(col_real_name))
    # 追加列：默认直接用列名
    return str(col_real_name)

dataset_real_cols = final_ds_cols
dataset_headers = [DATASET_DISPLAY.get(c, str(c)) for c in dataset_real_cols]

# tabularx 列格式：@{} l l l | l *{N}{Y} | Y @{}
n_y = max(0, len(dataset_headers) - 1)
colspec = f"@{{}}l l l |l *{{{n_y}}}{{Y}}|Y@{{}}"

label = f"table::{SETTING}_LCR{KEEP_LCR_VALUE:g}"

lines = []
lines.append("\\begin{table*}[ht!]")
lines.append("\\centering")
lines.append("\\tiny")
lines.append("\\setlength{\\tabcolsep}{4pt}")
lines.append("\\newcolumntype{Y}{>{\\centering\\arraybackslash}X}")
lines.append(f"\\begin{{tabularx}}{{\\textwidth}}{{{colspec}}}")
lines.append("\\toprule")

header_cells = ["Metric", "KCR", "Method"] + dataset_headers + ["\\textbf{Avg.}"]
lines.append(" & ".join(header_cells) + " \\\\ \\midrule")

metrics_in_table = [m for m in METRICS_MAP[SETTING] if m in pivot.index.get_level_values(0).unique()]
kcr_in_table = [k for k in KCR_ORDER if k in pivot.index.get_level_values(1).unique()]

total_cols = 3 + len(dataset_headers) + 1
cmid_l, cmid_r = 2, total_cols

for mi, metric in enumerate(metrics_in_table):
    blocks = []
    for kcr in kcr_in_table:
        methods_block = [m for m in method_order if (metric, kcr, m) in formatted.index]
        if methods_block:
            blocks.append((kcr, methods_block))
    if not blocks:
        continue

    metric_span = sum(len(ms) for _, ms in blocks)
    metric_written = False

    for bi, (kcr, methods_block) in enumerate(blocks):
        kcr_span = len(methods_block)
        kcr_written = False

        for method in methods_block:
            row_idx = (metric, kcr, method)
            row = formatted.loc[row_idx]

            metric_cell = f"\\multirow{{{metric_span}}}{{*}}{{{latex_escape_text(metric)}}}" if not metric_written else ""
            kcr_cell = f"\\multirow{{{kcr_span}}}{{*}}{{{fmt_kcr(kcr)}}}" if not kcr_written else ""
            metric_written = True
            kcr_written = True

            m_disp = latex_escape_text(method_display_name(method))

            vals = []
            for real_c in dataset_real_cols:
                v = row.get(real_c, "")
                vals.append(v if isinstance(v, str) else ("" if pd.isnull(v) else f"{v:.{ROUND_DECIMALS}f}"))

            vavg = row.get("__AVG__", "")
            avg_cell = vavg if isinstance(vavg, str) else ("" if pd.isnull(vavg) else f"{vavg:.{ROUND_DECIMALS}f}")

            lines.append(" & ".join([metric_cell, kcr_cell, m_disp] + vals + [avg_cell]) + " \\\\")

        if bi < len(blocks) - 1:
            lines.append(f"\\cmidrule(l){{{cmid_l}-{cmid_r}}}")

    if mi < len(metrics_in_table) - 1:
        lines.append("\\midrule")

lines.append("\\bottomrule")
lines.append("\\end{tabularx}")
lines.append(f"\\label{{{label}}}")
lines.append("\\end{table*}")

latex_table = "\n".join(lines)
Path(OUTPUT_TEX).write_text(latex_table, encoding="utf-8")
print(f"[OK] LaTeX written -> {Path(OUTPUT_TEX).resolve()}")


[OK] updated wide csv -> latex_or_csv/gcd/gcd_table_s31_plus.csv
[OK] LaTeX written -> /ssd/lijinpeng/code/bolt/latex_or_csv/gcd/gcd_table_s31.tex


/tmp/ipykernel_4037748/4005095200.py:332: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in num.groupby(level=[0, 1]):
/tmp/ipykernel_4037748/4005095200.py:332: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in num.groupby(level=[0, 1]):
/tmp/ipykernel_4037748/4005095200.py:332: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in num.